# Group Lab 6: Simple Environmental Modeling and Calibration

        **Week:** Week 12

        **Lab type:** Group lab

        **Estimated time:** 2 lab periods

        ## Learning objectives

        - Prepare model inputs.
- Run a simple water-balance model.
- Calibrate parameters.
- Evaluate model performance and limits.

        ## Earth and environmental motivation

        A simple water-balance model helps students connect precipitation, evapotranspiration, storage, and streamflow while seeing why calibration matters.

        ## Dataset

        Weather precipitation and streamflow

        ## Python concepts used

        - Water balance
- Storage
- Parameters
- Calibration
- Model-data comparison

## Group Lab 5 Debrief and Collaborative Debugging (First 10 Minutes)

Open the debrief card from Group Lab 5. Two to four students or groups will share a solved problem, an unresolved problem with evidence, or a verification choice. Work one unresolved problem together, then report to the class.

- 0-5 min: student discussion. Compare cards in small groups and
  debug one unresolved problem together.
- 5-10 min: student reports. Two to four groups report, and the
  class records one reusable lesson.


## Required imports and project paths

Run this cell first. It finds the project root whether the notebook is opened from the
repository root or from a notebook folder.

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
while not (PROJECT_ROOT / "data").exists() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_DIR = PROJECT_ROOT / "data"
PROCESSED_DIR = DATA_DIR / "processed"
SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

print(f"Project root: {PROJECT_ROOT}")
print(f"Processed data folder exists: {PROCESSED_DIR.exists()}")

## Group roles

- Module A: prepare precipitation, temperature, and streamflow data.
- Module B: implement and calibrate the model.
- Module C: evaluate performance and discuss limits.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

from earthcourse.modeling import calibrate_water_balance_grid_search, simple_water_balance_model
from earthcourse.stats import rmse

weather = pd.read_csv(PROCESSED_DIR / "iowa_city_weather_daily.csv", parse_dates=["date"])
stream = pd.read_csv(PROCESSED_DIR / "iowa_streamflow_daily.csv", parse_dates=["date"])
data = weather.merge(stream[["date", "discharge_cfs"]], on="date", how="inner").head(365)
precip = data["precipitation_mm"].to_numpy()
pet = (0.12 * data["temp_mean_c"].clip(lower=0)).to_numpy()
observed = data["discharge_cfs"].to_numpy() / data["discharge_cfs"].max() * 20

In [ ]:
best, table = calibrate_water_balance_grid_search(precip, pet, observed)
print(best)
sim = simple_water_balance_model(precip, pet, k_runoff=best["k_runoff"], k_baseflow=best["k_baseflow"])
print("Calibration RMSE:", rmse(observed, sim["simulated_flow_mm"]))

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(data["date"], observed, label="Observed scaled flow", color="black")
ax.plot(data["date"], sim["simulated_flow_mm"], label="Simulated flow", color="darkorange")
ax.set_xlabel("Date")
ax.set_ylabel("Flow index (mm/day)")
ax.set_title("Simple water-balance model calibration")
ax.legend()
fig.tight_layout()
plt.show()

## Guided coding: real units, real drainage area (Module A)

The quick version above scaled discharge to an arbitrary index. Real model
evaluation needs real units: divide river discharge by the watershed area to
get runoff depth in mm/day, the same unit as precipitation. The drainage
area comes from the multisite table; note `dtype={"site_no": str}`, which
stops pandas from silently turning the station ID "05454500" into the number
5454500.

In [ ]:
import numpy as np

sites = pd.read_csv(PROCESSED_DIR / "iowa_stream_sites_annual_flow.csv", dtype={"site_no": str})
iowa_city = sites[sites["site_no"].str.zfill(8) == "05454500"]
drainage_area_sqmi = float(iowa_city["drainage_area_sqmi"].iloc[0])
area_km2 = drainage_area_sqmi * 2.58999
print(f"Drainage area at Iowa City: {drainage_area_sqmi:,.0f} square miles = {area_km2:,.0f} km2")

data2 = weather.merge(stream[["date", "discharge_m3s"]], on="date", how="inner").head(730)
precip2 = data2["precipitation_mm"].to_numpy()
pet2 = (0.12 * data2["temp_mean_c"].clip(lower=0)).to_numpy()
observed_mm = data2["discharge_m3s"].to_numpy() * 86.4 / area_km2
print(f"Observed runoff range: {observed_mm.min():.2f} to {observed_mm.max():.2f} mm/day")

## Guided coding: calibrate on year 1, judge on year 2 (Module B)

Parameters tuned to one period always fit that period; the honest test is
the year the model never saw. Nash-Sutcliffe efficiency (NSE) is the
standard score: 1 is perfect, 0 means no better than predicting the mean,
and negative values mean worse than the mean. Do not expect a high NSE from
a one-bucket model without snow; explaining WHERE it fails is the deliverable.

In [ ]:
def nash_sutcliffe(observed, simulated):
    observed = np.asarray(observed, dtype=float)
    simulated = np.asarray(simulated, dtype=float)
    return 1.0 - np.sum((observed - simulated) ** 2) / np.sum((observed - observed.mean()) ** 2)

calibration_days = 365
best2, table2 = calibrate_water_balance_grid_search(
    precip2[:calibration_days], pet2[:calibration_days], observed_mm[:calibration_days]
)
print("Best parameters on the calibration year:",
      {k: round(v, 4) for k, v in best2.items()})

sim_all = simple_water_balance_model(
    precip2, pet2, k_runoff=best2["k_runoff"], k_baseflow=best2["k_baseflow"]
)
sim_flow = sim_all["simulated_flow_mm"].to_numpy()
nse_cal = nash_sutcliffe(observed_mm[:calibration_days], sim_flow[:calibration_days])
nse_val = nash_sutcliffe(observed_mm[calibration_days:], sim_flow[calibration_days:])
print(f"NSE, calibration year: {nse_cal:.2f}")
print(f"NSE, validation year:  {nse_val:.2f}")

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(data2["date"], observed_mm, color="black", linewidth=0.9, label="Observed runoff")
ax.plot(data2["date"], sim_flow, color="darkorange", linewidth=0.9, label="Simulated runoff")
split_date = data2["date"].iloc[calibration_days]
ax.axvline(split_date, color="gray", linestyle="--", linewidth=1)
ax.text(split_date, ax.get_ylim()[1] * 0.9, " validation starts", fontsize=9)
ax.set_xlabel("Date")
ax.set_ylabel("Runoff (mm/day)")
ax.set_title("Water-balance model in real units, calibration and validation")
ax.legend()
fig.tight_layout()
plt.show()

## Guided coding: which parameters matter? (Module B)

The grid search already scored every parameter pair; the calibration surface
below shows the full picture. A long flat valley means several parameter
pairs fit almost equally well, a situation modelers call equifinality, and
it is one reason a good fit alone never proves a model is right.

In [ ]:
pivot = table2.pivot(index="k_baseflow", columns="k_runoff", values="rmse")
fig, ax = plt.subplots(figsize=(7, 4.5))
image = ax.imshow(
    pivot.to_numpy(), origin="lower", aspect="auto", cmap="viridis",
    extent=[pivot.columns.min(), pivot.columns.max(), pivot.index.min(), pivot.index.max()],
)
ax.scatter(best2["k_runoff"], best2["k_baseflow"], color="red", marker="x", s=80, label="Best fit")
ax.set_xlabel("k_runoff (fraction of surplus to quickflow)")
ax.set_ylabel("k_baseflow (fraction of storage per day)")
ax.set_title("Calibration surface: RMSE across the parameter grid")
fig.colorbar(image, ax=ax, label="RMSE (mm/day)")
ax.legend()
fig.tight_layout()
plt.show()

## Optional responsible AI use

Use the workflow from Group Lab 4 to review a model function, add a sanity check, improve the figure, or improve documentation. Update `AI_USAGE_LOG.md` if used.

## Graded Checkpoint: Independent Analysis

The guided cells are examples. Complete the task below with your own code; an unchanged guided notebook does not meet the submission requirement.

Repeat the real-unit calibration with a PET coefficient of 0.20 instead of 0.12. Compare calibration and validation NSE with the original run, and name one process that the parameter change cannot represent.


In [ ]:
# GRADED CHECKPOINT
# Write your code below. Include at least one verification check.


### Scientific Explanation

Replace this text with your interpretation. State what the result means, cite one piece of numerical or graphical evidence, and name one limitation.


## More practice

1. Plot the simulated storage (`sim_all["storage_mm"]`) through both years.
   In which season does the bucket fill, and does that match your intuition
   for Iowa?
2. Average the residual (observed minus simulated) by month. Which months
   does the model miss most, and what missing process (snow accumulation and
   melt, frozen ground, crop water use) best explains the pattern?
3. Change the PET coefficient from 0.12 to 0.20, recalibrate, and compare
   NSE. What does the result say about how input uncertainty trades off
   against parameter values?

## Common mistakes and debugging tips

- Check that `PROCESSED_DIR.exists()` printed `True`.
- Read error messages from the bottom upward.
- Check column names with `df.columns` before selecting a column.
- Keep units in figure labels and written interpretations.
- Re-run earlier cells after changing data-loading or helper-code cells.

## Deliverables checklist

        - [ ] Prepared model inputs
- [ ] Calibrated model
- [ ] Observed vs simulated plot
- [ ] Model limitation discussion

        ## Short reflection

        Which model assumptions are most important to mention before using this result?

        ## Rubric summary

        Correctness and completion, readable code, labeled figures, interpretation,
        and reproducibility all matter. Your submitted notebook should run from top to bottom.

## Debrief Card for the Next Lab

Complete this before the next Wednesday meeting. An unresolved problem is a valid and useful report.

**Goal:** Replace this text.

**Expected result:** Replace this text.

**What happened:** Replace this text.

**Evidence:** Include an error message, value, figure observation, or tiny test.

**What I tried:** Replace this text.

**Fix or next check:** State what fixed it, or what the class should test next.

**Lesson from a classmate:** Complete this during the next debrief.
